# SQL Analytics for Customer Churn

In this notebook, we'll load our cleaned dataset into an SQLite database to simulate a data warehouse environment. We'll then execute analytical SQL queries to answer business questions using `pandas` and `sqlalchemy`.

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

## 1. Setup SQLite Database and Load Data

In [2]:
# Create SQLite engine
engine = create_engine('sqlite:///../data/churn_analytics.db')

# Load the cleaned dataset
df = pd.read_csv('../data/cleaned_indian_bank_churn.csv')
print(f"Loaded {len(df)} rows from CSV.")

# Write to SQLite
df.to_sql('customer_churn', con=engine, if_exists='replace', index=False)
print("Data loaded into SQLite table 'customer_churn' successfully.")

Loaded 10000 rows from CSV.
Data loaded into SQLite table 'customer_churn' successfully.


## 2. Execute Analytical SQL Queries
Let's run a few of the queries from our `sql/analytical_queries.sql` file.

### Query 1: Overall Churn Rate

In [3]:
query_1 = """
SELECT 
    COUNT(*) as total_customers,
    SUM(Churn) as churned_customers,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 2) as churn_rate_pct
FROM customer_churn;
"""
print("--- Overall Churn Rate ---")
display(pd.read_sql(query_1, con=engine))

--- Overall Churn Rate ---


,total_customers,churned_customers,churn_rate_pct
0,10000,1073,10.73


### Query 2: Churn Rate by Location (using CTE)

In [4]:
query_2 = """
WITH LocationStats AS (
    SELECT 
        Location,
        COUNT(*) as total,
        SUM(Churn) as churned
    FROM customer_churn
    GROUP BY Location
)
SELECT 
    Location,
    total,
    churned,
    ROUND((churned * 100.0) / total, 2) as churn_rate_pct
FROM LocationStats
ORDER BY churn_rate_pct DESC;
"""
print("--- Churn Rate by Location ---")
display(pd.read_sql(query_2, con=engine))

--- Churn Rate by Location ---


,Location,total,churned,churn_rate_pct
0,Kolkata,1062,129,12.15
1,Chennai,1139,137,12.03
2,Mumbai,2214,245,11.07
3,Pune,1123,122,10.86
4,Bangalore,1109,119,10.73
5,Delhi,2249,224,9.96
6,Hyderabad,1104,97,8.79


### Query 3: Average Balance and Salary by Churn Status

In [5]:
query_3 = """
SELECT 
    CASE WHEN Churn = 1 THEN 'Churned' ELSE 'Retained' END as Status,
    ROUND(AVG(Balance), 2) as avg_balance,
    ROUND(AVG(EstimatedSalary), 2) as avg_salary,
    COUNT(*) as customer_count
FROM customer_churn
GROUP BY Churn;
"""
print("--- Financials by Churn Status ---")
display(pd.read_sql(query_3, con=engine))

--- Financials by Churn Status ---


,Status,avg_balance,avg_salary,customer_count
0,Retained,64093.85,85563.16,8927
1,Churned,73436.75,85147.92,1073


### Query 4: Identifying High-Risk Segments (Age > 50 and Inactive)

In [6]:
query_4 = """
SELECT 
    Gender,
    COUNT(*) as segment_size,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_pct
FROM customer_churn
WHERE Age > 50 AND IsActiveMember = 0
GROUP BY Gender
ORDER BY churn_rate_pct DESC;
"""
print("--- High-Risk Segment (Age>50 & Inactive) ---")
display(pd.read_sql(query_4, con=engine))

--- High-Risk Segment (Age>50 & Inactive) ---


,Gender,segment_size,churn_rate_pct
0,Male,496,23.19
1,Female,431,20.65


### Query 5: Churn by Number of Products (Window Functions)

In [7]:
query_5 = """
SELECT 
    NumOfProducts,
    COUNT(*) as customer_count,
    SUM(Churn) as churned,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_pct,
    SUM(COUNT(*)) OVER (ORDER BY NumOfProducts) as running_total_customers
FROM customer_churn
GROUP BY NumOfProducts;
"""
print("--- Churn by Products ---")
display(pd.read_sql(query_5, con=engine))

--- Churn by Products ---


,NumOfProducts,customer_count,churned,churn_rate_pct,running_total_customers
0,1,4915,534,10.86,4915
1,2,4485,374,8.34,9400
2,3,404,113,27.97,9804
3,4,196,52,26.53,10000
